# 🛠️ Build a Complete RAG System From Scratch

This is the capstone tutorial. We build a full RAG system step by step,
combining everything from this course into one working pipeline.

## What We'll Build

```
1. Load + chunk documents
2. Embed + index chunks
3. Accept a user question
4. Search for relevant chunks
5. Re-rank results
6. Build a grounded prompt
7. Call an LLM
8. Return answer with citations
```

By the end you'll have a reusable RAG class you can drop into any project.

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


In [2]:
# !pip install sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')   # fast retrieval
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # re-ranking

print("Models loaded!")

Models loaded!


## Step 1 — Load and Chunk Documents

In [4]:
# Your documents — replace with real files
raw_text = """
RAG stands for Retrieval Augmented Generation. It combines document retrieval
with language model generation to answer questions from your own data.

Embeddings are dense vector representations of text. Similar texts have
similar vectors, measured by cosine similarity.

Chunking splits long documents into smaller pieces. A common strategy is
overlapping chunks — each chunk shares some text with the next to preserve context.

Reranking uses a cross-encoder to re-score retrieved documents more accurately
than bi-encoder retrieval alone. It is slower but improves precision.

Evaluation of RAG uses metrics like faithfulness (is the answer grounded in
the sources?) and relevance (does it answer the question?).
"""

# Simple sentence chunker
sentences = [s.strip() for s in raw_text.split('\n') if len(s.strip()) > 20]

# Build overlapping chunks of 2 sentences
chunks = []
for i in range(len(sentences)):
    chunk_text = " ".join(sentences[i:i+2])
    chunks.append({"id": i, "text": chunk_text})

print(f"Created {len(chunks)} chunks")
for c in chunks[:3]:
    print(f"  [{c['id']}] {c['text'][:70]}...")

Created 10 chunks
  [0] RAG stands for Retrieval Augmented Generation. It combines document re...
  [1] with language model generation to answer questions from your own data....
  [2] Embeddings are dense vector representations of text. Similar texts hav...


## Step 2 — Embed and Index

In [5]:
texts      = [c["text"] for c in chunks]
embeddings = bi_encoder.encode(texts)

print(f"Indexed {len(chunks)} chunks — {embeddings.shape[1]}-dim vectors")

Indexed 10 chunks — 384-dim vectors


## Step 3 — Retrieve

In [6]:
question = "What is reranking and why is it useful?"

# Bi-encoder retrieval — fast, gets top-K candidates
q_emb  = bi_encoder.encode(question)
scores = cosine_similarity([q_emb], embeddings)[0]
top_k  = np.argsort(scores)[::-1][:5]   # retrieve top 5 candidates

candidates = [chunks[i] for i in top_k]

print("Top 5 candidates (bi-encoder):")
for i, c in enumerate(candidates, 1):
    print(f"  {i}. [{scores[top_k[i-1]]:.3f}] {c['text'][:70]}...")

Top 5 candidates (bi-encoder):
  1. [0.677] Reranking uses a cross-encoder to re-score retrieved documents more ac...
  2. [0.612] overlapping chunks — each chunk shares some text with the next to pres...
  3. [0.245] similar vectors, measured by cosine similarity. Chunking splits long d...
  4. [0.225] the sources?) and relevance (does it answer the question?)....
  5. [0.191] Embeddings are dense vector representations of text. Similar texts hav...


## Step 4 — Re-rank

In [7]:
# Cross-encoder re-ranks the top-5 for better precision
pairs  = [[question, c["text"]] for c in candidates]
rerank_scores = cross_encoder.predict(pairs)

# Sort candidates by rerank score
ranked = sorted(zip(rerank_scores, candidates), reverse=True)
top_2  = [c for _, c in ranked[:2]]   # keep top 2 after reranking

print("Top 2 after reranking (cross-encoder):")
for i, (score, c) in enumerate(ranked[:2], 1):
    print(f"  {i}. [{score:.3f}] {c['text'][:70]}...")

Top 2 after reranking (cross-encoder):
  1. [5.698] Reranking uses a cross-encoder to re-score retrieved documents more ac...
  2. [3.535] overlapping chunks — each chunk shares some text with the next to pres...


## Step 5 — Build Prompt and Generate

In [8]:
# Build the grounded prompt
context = "\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(top_2))

prompt = f"""Answer the question using ONLY the sources below.
Cite sources as [1], [2], etc.

Sources:
{context}

Question: {question}
Answer:"""

print(prompt)

Answer the question using ONLY the sources below.
Cite sources as [1], [2], etc.

Sources:
[1] Reranking uses a cross-encoder to re-score retrieved documents more accurately than bi-encoder retrieval alone. It is slower but improves precision.
[2] overlapping chunks — each chunk shares some text with the next to preserve context. Reranking uses a cross-encoder to re-score retrieved documents more accurately

Question: What is reranking and why is it useful?
Answer:


In [9]:
# Send the grounded prompt to Claude
answer = llm(prompt, max_tokens=300)

print(f"Q: {question}")
print(f"A: {answer}")
print("\nSources used:")
for i, c in enumerate(top_2, 1):
    print(f"  [{i}] {c['text'][:70]}...")


Q: What is reranking and why is it useful?
A: Reranking is a technique that uses a cross-encoder to re-score retrieved documents more accurately than bi-encoder retrieval alone [1]. 

It is useful because it improves precision [1], though it comes with a tradeoff of being slower [1].

Sources used:
  [1] Reranking uses a cross-encoder to re-score retrieved documents more ac...
  [2] overlapping chunks — each chunk shares some text with the next to pres...


## The Complete Pipeline — All Together

In [10]:
class RAGSystem:
    """Complete RAG system: load → index → retrieve → rerank → generate."""

    def __init__(self, documents):
        self.chunks     = self._chunk(documents)
        self.embeddings = bi_encoder.encode([c["text"] for c in self.chunks])
        print(f"Indexed {len(self.chunks)} chunks")

    def _chunk(self, text, min_len=20):
        sentences = [s.strip() for s in text.split('\n') if len(s.strip()) > min_len]
        return [{"id": i, "text": " ".join(sentences[i:i+2])} for i in range(len(sentences))]

    def ask(self, question, top_k=5, final_k=2):
        # 1. Retrieve
        q_emb     = bi_encoder.encode(question)
        scores    = cosine_similarity([q_emb], self.embeddings)[0]
        top_idx   = np.argsort(scores)[::-1][:top_k]
        candidates = [self.chunks[i] for i in top_idx]

        # 2. Re-rank
        pairs      = [[question, c["text"]] for c in candidates]
        rr_scores  = cross_encoder.predict(pairs)
        ranked     = sorted(zip(rr_scores, candidates), reverse=True)
        top_chunks = [c for _, c in ranked[:final_k]]

        # 3. Build prompt
        context = "\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(top_chunks))
        prompt  = f"Answer using only these sources. Cite [1],[2].\n\nSources:\n{context}\n\nQ: {question}\nA:"

        # 4. Generate answer with Claude
        answer = llm(prompt, max_tokens=300)
        return {"answer": answer, "sources": top_chunks}


# Use it!
rag    = RAGSystem(raw_text)
result = rag.ask("How do embeddings measure similarity?")

print("Sources retrieved:")
for i, c in enumerate(result["sources"], 1):
    print(f"  [{i}] {c['text'][:70]}...")

Indexed 10 chunks
Sources retrieved:
  [1] Embeddings are dense vector representations of text. Similar texts hav...
  [2] with language model generation to answer questions from your own data....


## Congratulations! 🎉

You just built a production-quality RAG system with:

```
✅ Document chunking
✅ Bi-encoder retrieval  
✅ Cross-encoder reranking
✅ Grounded prompting with citations
✅ LLM-ready output
```

## What to Do Next

1. **Swap in real documents** — point it at your own `.txt` or `.pdf` files
2. **Add a real LLM** — replace the simulated answer with an API call
3. **Add advanced search** — try HyDE or multi-query from Section 3
4. **Evaluate it** — use RAGAS or the metrics from Section 6
5. **Deploy it** — wrap in FastAPI from Section 7

You now understand the full RAG stack. Go build something! 🚀